### Appendix: LLM Annotation
#### Can LLM Annotation Serve as Gold Standard? 
Evaluating Reliability, Consistency, and Trustworthiness for Auditing Political Content on Social Media

---

Using multiple LLMs in a zero-shot setting to annotate a sample of 300 tweets across **8 anti-democratic attitude variables** (V1–V8) derived from the AAPA scale (Jia et al., 2024), 

This pipeline follows best practices outlined in:
- Törnberg, P. (2024). *How to Use Large-Language Models for Text Analysis*. arXiv:2307.13106
- Törnberg, P. (2024). *Best Practices for Text Annotation with Large Language Models*. Sociologica.

### Models Used
| LLM_ID | Model | Access | Rationale |
|--------|-------|--------|-----------|
| 2 | `claude-3-5-sonnet-20241022` | Anthropic API | Architectural diversity; Constitutional AI design relevant to democratic values critique |


### Output
Results are saved to `csv/llm_annotations_anthropic.csv`. 

## 0. Setup & Installation

Run this cell once to install required packages.

In [20]:
# Install required packages
# %pip install openai anthropic together pandas tqdm krippendorff scipy matplotlib seaborn

In [21]:
import sys
import os
import time
import json
import re
import math
from datetime import datetime

import pandas as pd
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv
from anthropic import Anthropic


## 1. Imports & Configuration

In [22]:
 
# API Keys
# Set these as environment variables rather than hardcoding in the notebook
load_dotenv() 

# Load key
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# Create client
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)



# ── Model Registry ───────────────────────────────────────────────────────────
# CRITICAL: Pin exact version strings for reproducibility.
# Models update silently; the version string is your audit trail.
MODELS = {
    2: {
        "llm_name_version": "claude-haiku-4-5-20251001",
        "provider":         "anthropic",
        "display_name":     "claude-haiku-4-5",
        "rationale":        "not sure yet"
    },
}

# File Paths
INPUT_FILE  = "csv/sampled_tweets_300.csv"   # sampled_tweets_300.csv is a random sample of 300 tweets from the full dataset, used for annotation.
OUTPUT_FILE = "csv/llm_annotations_anthropic.csv"      # Results — llm_annotations.csv will contain the original tweet text, the assigned label, and the rationale for each model's annotation.
CHECKPOINT  = "checkpoint_anthropic.json"          # Progress tracking — allows resuming if interrupted

# Annotation Parameters
TEMPERATURE    = 0      # Must be 0 for deterministic, reproducible output
MAX_RETRIES    = 10     # Retry limit per API call
RETRY_WAIT     = 15     # Seconds to wait between retries
REQUEST_DELAY  = 1.5    # Seconds between requests (rate limiting)

print("Configuration loaded.")
print(f"   Input:  {INPUT_FILE}")
print(f"   Output: {OUTPUT_FILE}")
print(f"   Models: {[m['display_name'] for m in MODELS.values()]}")
print(f"   Temperature: {TEMPERATURE} (deterministic)")

Configuration loaded.
   Input:  csv/sampled_tweets_300.csv
   Output: csv/llm_annotations_anthropic.csv
   Models: ['claude-haiku-4-5']
   Temperature: 0 (deterministic)


In [23]:
# get current working directory
print(os.getcwd()) 

/Users/mariameshi/Documents/year_5/Thesis/Code/LLM_annotations


## 2. Load & Inspect Dataset

In [24]:
df = pd.read_csv(INPUT_FILE)

# Ensure required columns exist
required_cols = ["tweet_id", "post_type", "text"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Loaded {len(df)} tweets")
print(f"\nColumn overview")
print(df.dtypes)
print(f"\nPost type distribution (1=original, 2=reply, 3=retweet):")
print(df['post_type'].value_counts())
print(f"\nSample tweets:")
df[["tweet_id", "post_type", "text"]].head(3)

Loaded 301 tweets

Column overview
tweet_id     int64
post_type      str
text           str
dtype: object

Post type distribution (1=original, 2=reply, 3=retweet):
post_type
reply    201
tweet     60
quote     40
Name: count, dtype: int64

Sample tweets:


,tweet_id,post_type,text
0,1,reply,@varadmehta LOL. Yet Matt Gaetz is a MAGA hero .
1,2,reply,@evan_kapitansky You are right. Love from a co...
2,3,tweet,"In my opinion, The Trump administration could ..."


In [25]:
# Text length distribution 
df['text_length'] = df['text'].str.len()
print("Tweet length statistics (characters)")
print(df['text_length'].describe().round(1))


Tweet length statistics (characters)
count    301.0
mean     163.1
std       96.3
min       12.0
25%       80.0
50%      151.0
75%      246.0
max      754.0
Name: text_length, dtype: float64


## 3. Codebook & Prompt Construction

Following Törnberg (2024) best practices:
- The system prompt establishes role and task context
- The full AAPA codebook is provided inline (not summarised)
- Output format is strictly specified to enable reliable parsing
- Temperature = 0 ensures determinism


In [26]:
# System Prompt
# Establishes annotator role and grounding in political science literature.
# Based on Jia et al. (2024) AAPA codebook, itself derived from:
# Kingzette et al. (2021), Voelkel et al. (2023) and the AAPA scale.

SYSTEM_PROMPT = """You are an expert political content annotator trained in political science. 
Your task is to annotate social media posts (tweets) from the United States for anti-democratic 
attitudes and partisan animosity. You will apply a structured codebook derived from the 
Anti-democratic Attitudes and Partisan Animosity (AAPA) scale used in academic research 
(Jia et al., 2024; Voelkel et al., 2023).

You must rate each tweet on EIGHT variables (V1–V8), each on a scale of 1 to 3.
For each variable, you will provide:
  - A numeric rating (1, 2, or 3)
  - A reason citing specific language in the tweet

You MUST respond ONLY with a valid JSON object. Do not include any text before or after the JSON.
Do not include markdown code fences. Return only the raw JSON."""

# Codebook
# Reproduced in full from the dissertation codebook, Jia et al. (2024) to ensure fidelity to the original annotation scheme.
# Summarising the codebook risks introducing annotation drift, which would undermine the validity of our comparisons to human annotations and prior research.

CODEBOOK = """
CODEBOOK — Anti-Democratic Attitudes and Partisan Animosity Scale
================================================================

V1: PARTISAN ANIMOSITY
Partisan animosity is defined as "dislike for opposing partisans". 
Factors:
  A: Partisan name-calling
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V2: SUPPORT FOR UNDEMOCRATIC PRACTICES
Support for undemocratic practices is defined as "willingness to forgo democratic principles for partisan gain". 
Undemocratic practices are undemocratic tendencies or actions such as reducing polling stations in areas that support their opponents, attacking the independence of the judiciary, undermining the free press, and challenging the legitimacy of election results, and encouraging political violence. 
Factors:
  A: Shows support for undemocratic practices
  B1: Partisan name-calling
  B2: Emotion or exaggeration
Ratings:
  1 = Does not satisfy any factor
  2 = Does not satisfy A, but satisfies B1 or B2
  3 = Satisfies A, B1, and B2

V3: SUPPORT FOR PARTISAN VIOLENCE
Support for partisan violence is defined as "willingness to use violent tactics against outpartisans". 
Examples of partisan violence include sending threatening and intimidating messages to the opponent party, harassing the opponent party on the Internet, using violence in advancing their political goals or winning more races in the next election.
Factors:
  A: Shows support for partisan violence
  B1: Partisan name-calling
  B2: Emotion or exaggeration
Ratings:
  1 = Does not satisfy any factor
  2 = Does not satisfy A, but satisfies B1 or B2
  3 = Satisfies A, B1, and B2

V4: SUPPORT FOR UNDEMOCRATIC CANDIDATES
Support for undemocratic candidates is defined as "willingness to ignore undemocratic practices to elect in party candidates". 
Undemocratic candidates are oftentimes those who support the following undemocratic practices such as reducing polling stations in areas that support their opponents, attacking the independence of the judiciary, undermining the free press, and challenging the legitimacy of election results, and encouraging political violence .
Factors:
  A: Shows support for undemocratic candidates
  B1: Partisan name-calling
  B2: Emotion or exaggeration
Ratings:
  1 = Does not satisfy any factor
  2 = Satisfies A, but not B1 or B2
  3 = Satisfies A, B1, and B2

V5: OPPOSITION TO BIPARTISANSHIP
Opposition to bipartisanship is defined as "resistance to cross-partisan collaboration". 
Factors:
  A: Any name-calling or terms that reduce trust
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V6: SOCIAL DISTRUST
Social distrust is defined as "distrust of people in general". 
Factors:
  A: Any name-calling or terms that reduce trust
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V7: SOCIAL DISTANCE
Social distance is defined as "resistance to interpersonal contact with outpartisans".
Factors:
  A: Terms that increase distrust, distance, insecurity, hate, prejudice, or discrimination
  B1: Emotion or exaggeration
  B2: Events that damage communities or decrease societal trust (e.g. mass shooting)
Ratings:
  1 = Does not satisfy any factor
  2 = Satisfies A, but not B1 or B2
  3 = Satisfies A, and B1 or B2

V8: BIASED EVALUATION OF POLITICISED FACTS
Biased evaluation of politicized facts is defined as "skepticism of facts that favor the worldview of the other party". 
Factors:
  A: Partially presents political facts or discusses controversial issue with partisan stance
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist
"""

# Output Format Specification
OUTPUT_FORMAT = """
Respond ONLY with a JSON object in exactly this format (no extra fields, no markdown):
{
  "V1": {"score": <1|2|3>, "V1_reason": "<brief reason citing tweet language>"},
  "V2": {"score": <1|2|3>, "V2_reason": "<brief reason citing tweet language>"},
  "V3": {"score": <1|2|3>, "V3_reason": "<brief reason citing tweet language>"},
  "V4": {"score": <1|2|3>, "V4_reason": "<brief reason citing tweet language>"},
  "V5": {"score": <1|2|3>, "V5_reason": "<brief reason citing tweet language>"},
  "V6": {"score": <1|2|3>, "V6_reason": "<brief reason citing tweet language>"},
  "V7": {"score": <1|2|3>, "V7_reason": "<brief reason citing tweet language>"},
  "V8": {"score": <1|2|3>, "V8_reason": "<brief reason citing tweet language>"}
}
"""

def build_user_prompt(tweet_text: str, post_type: str) -> str:
    """Construct the per-tweet user prompt."""
    return f"""{CODEBOOK}
{OUTPUT_FORMAT}

Now annotate the following {post_type}:

\"{tweet_text}\""""

# Preview the prompt for one tweet
example_tweet = df.iloc[0]
print("SYSTEM PROMPT (first 200 chars) preview:")
print(SYSTEM_PROMPT[:200], "...")
print("\nUSER PROMPT preview (first 400 chars) for the first tweet:")
print(build_user_prompt(example_tweet['text'], example_tweet['post_type'])[:400], "...")

SYSTEM PROMPT (first 200 chars) preview:
You are an expert political content annotator trained in political science. 
Your task is to annotate social media posts (tweets) from the United States for anti-democratic 
attitudes and partisan ani ...

USER PROMPT preview (first 400 chars) for the first tweet:

CODEBOOK — Anti-Democratic Attitudes and Partisan Animosity Scale

V1: PARTISAN ANIMOSITY
Partisan animosity is defined as "dislike for opposing partisans". 
Factors:
  A: Partisan name-calling
  B: Emotion or exaggeration
Ratings:
  1 = Neither A nor B exists
  2 = One of A or B exists
  3 = Both A and B exist

V2: SUPPORT FOR UNDE ...


## 4. OpenAI API Call

- Set temperature = 0
- Implement retry logic with exponential-style backoff
- Return a parsed JSON dict or raise after max retries

In [27]:
def parse_json_response(raw_text: str) -> dict:
    """
    Safely extract and parse JSON from a model response.
    Handles cases where the model wraps the JSON in markdown code fences
    despite instructions not to — a common failure mode.
    """
    # Strip markdown fences if present
    cleaned = re.sub(r"```(?:json)?\s*", "", raw_text).strip().rstrip("`").strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse JSON.\nRaw response:\n{raw_text}\nError: {e}")


def validate_annotation(annotation: dict) -> dict:
    """
    Validate that all 8 variables are present and scores are in range [1, 2, 3].
    Returns the validated annotation or raises ValueError.
    """
    for v in [f"V{i}" for i in range(1, 9)]:
        if v not in annotation:
            raise ValueError(f"Missing variable {v} in annotation")
        score = annotation[v].get("score")
        if score not in [1, 2, 3]:
            raise ValueError(f"Invalid score for {v}: {score} (must be 1, 2, or 3)")
    return annotation


# Anthropic (Claude Haiku 4.5)
def annotate_anthropic(tweet_text: str, post_type: str,
                        model: str = "claude-haiku-4-5-20251001") -> dict:
    """
    Annotate a tweet using the Anthropic API.
    Note: Anthropic does not have a JSON model: rely on the prompt + parsing.
    """
    for attempt in range(MAX_RETRIES):
        try:
            response = anthropic_client.messages.create(
                model=model,
                max_tokens=1024,
                temperature=TEMPERATURE,
                system=SYSTEM_PROMPT,
                messages=[
                    {"role": "user", "content": build_user_prompt(tweet_text, post_type)}
                ]
            )
            raw = response.content[0].text
            annotation = parse_json_response(raw)
            return validate_annotation(annotation)

        except Exception as e:
            print(f"  [Anthropic] Attempt {attempt+1}/{MAX_RETRIES} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_WAIT * (attempt + 1))
            else:
                raise RuntimeError(f"Anthropic failed after {MAX_RETRIES} attempts: {e}")


# Dispatcher to route to the correct annotator based on post type
ANNOTATORS = {
    1: annotate_anthropic,
}

print("ANTHROPIC API function defined.")
print("Providers:", [MODELS[i]['display_name'] for i in ANNOTATORS])

ANTHROPIC API function defined.
Providers: ['claude-haiku-4-5']


## 5. Main Annotation Loop

This section runs the full annotation pipeline across all tweets and all models.

**Key design decisions (following Törnberg, 2024):**
- **Checkpointing:** Progress is saved after every tweet. If the notebook crashes or the API rate-limits, re-running the cell picks up where it left off.
- **Sequential by model:** Annotating all tweets for Model 1 before moving to Model 2 simplifies cost tracking and debugging.
- **Annotation_Count tracking:** Records how many times each tweet has been annotated (useful if you later re-annotate for stability checks).

In [28]:
# Output Schema
OUTPUT_COLUMNS = [
    "llm_annotation_id", "tweet_id", "llm_id", "llm_name_version", "post_type", "text",
    "V1", "V1_reason",
    "V2", "V2_reason",
    "V3", "V3_reason",
    "V4", "V4_reason",
    "V5", "V5_reason",
    "V6", "V6_reason",
    "V7", "V7_reason",
    "V8", "V8_reason",
    "total_score", "date", "annotation_session"
]

SCORE_COLS = ["V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8"]


def load_checkpoint() -> set:
    """Load set of (tweet_id, llm_id) pairs already completed."""
    if os.path.exists(CHECKPOINT):
        with open(CHECKPOINT, "r") as f:
            data = json.load(f)
        completed = set(tuple(x) for x in data.get("completed", []))
        print(f"✅ Checkpoint loaded: {len(completed)} annotations already done.")
        return completed
    return set()


def save_checkpoint(completed: set):
    """Save progress to checkpoint file."""
    with open(CHECKPOINT, "w") as f:
        json.dump({"completed": [list(x) for x in completed]}, f)


def annotation_to_row(tweet_row: pd.Series, llm_id: int,
                       annotation: dict, annotation_id: int,
                       annotation_session: str) -> dict:
    """Convert a raw API annotation dict into a flat output row."""
    model_cfg = MODELS[llm_id]
    scores  = {v: annotation[v]['score']            for v in annotation}
    reasons = {v: annotation[v][f'{v}_reason']      for v in annotation}
    total     = sum(scores.values())

    row = {
        "llm_annotation_id":  annotation_id,
        "tweet_id":           tweet_row['tweet_id'],
        "llm_id":             llm_id,
        "llm_name_version":   model_cfg['llm_name_version'],
        "post_type":          tweet_row['post_type'],
        "text":               tweet_row['text'],
        "total_score":        total,
        "date":               datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "annotation_session": annotation_session,
    }

    for v in SCORE_COLS:
        row[v]               = scores[v]
        row[f"{v}_reason"]   = reasons[v]

    return row


print("Output schema and helper functions defined.")

Output schema and helper functions defined.


In [29]:
# Run this on a single tweet to inspect the raw output
test_tweet = df.iloc[0]
response = annotate_anthropic(test_tweet['text'], test_tweet['post_type'])
print(response)

{'V1': {'score': 2, 'V1_reason': "Contains partisan name-calling ('MAGA hero') with sarcasm/emotion ('LOL') directed at opposing partisan, satisfying one factor (B: emotion/exaggeration)."}, 'V2': {'score': 1, 'V2_reason': 'Does not show support for undemocratic practices; merely expresses partisan sentiment about a political figure.'}, 'V3': {'score': 1, 'V3_reason': 'Does not show support for partisan violence or violent tactics.'}, 'V4': {'score': 1, 'V4_reason': 'Does not show support for undemocratic candidates; only expresses sarcastic partisan opinion.'}, 'V5': {'score': 1, 'V5_reason': 'Does not express resistance to cross-partisan collaboration; is a brief partisan jab without opposing bipartisanship explicitly.'}, 'V6': {'score': 1, 'V6_reason': 'Does not express distrust of people in general; targets a specific political figure.'}, 'V7': {'score': 1, 'V7_reason': 'Does not express resistance to interpersonal contact with outpartisans or terms increasing distrust/distance tow

In [30]:
# Run the full Pipeline
# Safe to interrupt and re-run as checkpointing prevents duplicate work.
#
#   Cost estimate (approximate, early 2025):
#   GPT-4o:  300 tweets × ~800 tokens in + ~300 out ≈ $3–5 total

ANNOTATION_SESSION = datetime.now().strftime("%Y%m%d_%H%M")  # e.g. "20250225_1430"

completed     = load_checkpoint()
all_results   = []
annotation_id = 1

if os.path.exists(OUTPUT_FILE):
    existing      = pd.read_csv(OUTPUT_FILE)
    all_results   = existing.to_dict('records')
    annotation_id = len(all_results) + 1
    print(f"Resuming: {len(all_results)} rows already in {OUTPUT_FILE}")

total_combinations = len(df) * len(MODELS)
print(f"\nTotal to complete: {total_combinations} | Done: {len(completed)} | Remaining: {total_combinations - len(completed)}\n")


with tqdm(total=total_combinations - len(completed), desc="Annotating") as pbar:
    for llm_id, model_cfg in MODELS.items():
        annotate_fn = ANNOTATORS[llm_id]
        print(f"\n── Model: {model_cfg['display_name']} ({model_cfg['llm_name_version']}) ──")

        for _, tweet_row in df.iterrows():
            tweet_id = tweet_row['tweet_id']
            key      = (tweet_id, llm_id)

            if key in completed:
                continue

            try:
                annotation = annotate_fn(
                    tweet_text=tweet_row['text'],
                    post_type=tweet_row['post_type']
                )
                result_row = annotation_to_row(
                    tweet_row, llm_id, annotation,
                    annotation_id, ANNOTATION_SESSION
                )
                all_results.append(result_row)
                annotation_id += 1

                completed.add(key)
                save_checkpoint(completed)

                if len(all_results) % 10 == 0:
                    pd.DataFrame(all_results, columns=OUTPUT_COLUMNS).to_csv(OUTPUT_FILE, index=False)

                time.sleep(REQUEST_DELAY)
                pbar.update(1)

            except Exception as e:
                print(f"    Failed: tweet {tweet_id}, model {llm_id}: {e}")
                continue

# Final save
results_df = pd.DataFrame(all_results, columns=OUTPUT_COLUMNS)
results_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nRating finished. {len(results_df)} rows saved to '{OUTPUT_FILE}'.")


Total to complete: 301 | Done: 0 | Remaining: 301



Annotating:   0%|          | 0/301 [00:00<?, ?it/s]


── Model: claude-haiku-4-5 (claude-haiku-4-5-20251001) ──


Annotating: 100%|██████████| 301/301 [24:48<00:00,  4.95s/it]


Rating finished. 301 rows saved to 'csv/llm_annotations_anthropic.csv'.


---

## References

- Jia, C., Lam, M. S., Mai, M. C., Hancock, J. T., & Bernstein, M. S. (2024). Embedding Democratic Values into Social Media AIs via Societal Objective Functions. *PACMHCI CSCW1*.
- Törnberg, P. (2023). ChatGPT-4 Outperforms Experts and Crowd Workers in Annotating Political Twitter Messages with Zero-Shot Learning. *arXiv:2304.06588*.
- Törnberg, P. (2023). How to Use Large-Language Models for Text Analysis. *arXiv:2307.13106*.
- Törnberg, P. (2024). Best Practices for Text Annotation with Large Language Models. *Sociologica*.
- Törnberg, P. (2024/2025). Large Language Models Outperform Expert Coders and Supervised Classifiers at Annotating Political Social Media Messages. *Social Science Computer Review*.
- Gilardi, F., Alizadeh, M., & Kubli, M. (2023). ChatGPT Outperforms Crowd Workers for Text-Annotation Tasks. *PNAS*.
- Krippendorff, K. (2018). *Content Analysis: An Introduction to Its Methodology* (4th ed.).
- USE24-XD (2025). Wisdom of the LLM Crowd. *arXiv:2602.11962*.
- https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/claude-prompting-best-practices

---
*Notebook version: 1.0 | Temperature: 0 (deterministic) | Models pinned to exact version strings above*

---
**Reproducibility Note:** Temperature is set to `0` for all models to ensure deterministic reproducable output. 
!Record exact model version strings, LLMs update silently and results may differ across versions.!